## Mistral 7B v0.2

In [2]:
from transformers import AutoTokenizer, MistralForCausalLM

trained_weights = "mistralai/Mistral-7B-Instruct-v0.2"

model = MistralForCausalLM.from_pretrained(trained_weights)
tokenizer = AutoTokenizer.from_pretrained(trained_weights)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [3]:
# https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2 
# https://mistral.ai/news/announcing-mistral-7b

prompt = "Hey, what is a finance bro?"
inputs = tokenizer(prompt, return_tensors="pt")

generate_ids = model.generate(inputs.input_ids, max_length = 30)
tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


'Hey, what is a finance bro?\n\nA finance broker is a professional who helps individuals and businesses secure financing for various purposes, such as'

In [5]:
prompt = " Classify this phrase as positive, negative or neutral: The food was good, but not amazing."
inputs = tokenizer(prompt, return_tensors="pt")

generate_ids = model.generate(inputs.input_ids, max_length = 100)
tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'Classify this phrase as positive, negative or neutral: The food was good, but not amazing.\n\nAnswer: Neutral. The phrase acknowledges that the food was satisfactory, but it did not elicit a strong positive response.'

In [6]:
prompt = " Classify the market reaction to this phrase as positive, negative or neutral: US has hit more than 1,700 targets in Iran so far, officials say."
inputs = tokenizer(prompt, return_tensors="pt")

generate_ids = model.generate(inputs.input_ids, max_length = 100)
tokenizer.batch_decode(generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


'Classify the market reaction to this phrase as positive, negative or neutral: US has hit more than 1,700 targets in Iran so far, officials say.\n\nNeutral: The statement is factual and does not contain any positive or negative connotation. It simply reports the number of targets hit by the US in Iran, without expressing an opinion or making a judgment. The market reaction to this statement would likely be neutral as well, as it does not provide'

In [9]:
import gc
import torch

# # Example: you used these
# # model, tokenizer, optimizer, lr_scheduler, train_dataloader, etc.

del model
del tokenizer  # if on GPU or large
# del optimizer
# del lr_scheduler
# del train_dataloader   # if it holds GPU tensors, optional

gc.collect()              # force Python garbage collection
torch.cuda.empty_cache()  # release unused cached VRAM


NameError: name 'model' is not defined

In [10]:
gc.collect()              # force Python garbage collection
torch.cuda.empty_cache()

# LoRA (Low-Rank Adaptation) fine-tuning

- https://huggingface.co/learn/llm-course/chapter11/4
- https://huggingface.co/mistralai/Mistral-7B-v0.1
- https://huggingface.co/mistralai/Mistral-7B-v0.3
- https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.2 <--

## Dataset Prep

In [2]:
from datasets import load_dataset

ds = load_dataset("virattt/financial-qa-10K")

In [3]:
df2 = ds["train"].to_pandas()
df2.head()

,question,answer,context,ticker,filing
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha...",NVDA,2023_10K
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...,NVDA,2023_10K
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...,NVDA,2023_10K
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"NVIDIA has a platform strategy, bringing toget...",NVDA,2023_10K
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,With our introduction of the CUDA programming ...,NVDA,2023_10K


In [4]:
# FORMAT 1: Teaches the model to use a context to answer (using RAG later)
format_1 = """Use the following context to answer the question.

Context: {context}

Question: {question}
"""

# FORMAT 2 : When no context is available
format_2 = """ {question} """


In [5]:
# example
question = df2.question[0]
context = df2.context[0]
answer = df2.answer[0]


In [6]:
# create dataset to perform the training
# using the correct format
# https://huggingface.co/docs/transformers/main/chat_templating
from transformers import AutoTokenizer, MistralForCausalLM

trained_weights = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(trained_weights)


In [7]:
content = format_1.format(context = context, question=question)
print(content)

chat = [{"role": "user", "content": content},
        {"role": "assistant", "content": answer}]
tokenizer.apply_chat_template(chat, tokenize = False)

Use the following context to answer the question.

Context: Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.

Question: What area did NVIDIA initially focus on before expanding to other computationally intensive fields?



'<s> [INST] Use the following context to answer the question.\n\nContext: Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.\n\nQuestion: What area did NVIDIA initially focus on before expanding to other computationally intensive fields?\n [/INST] NVIDIA initially focused on PC graphics.</s>'

In [8]:
def format_row(row, tokenizer, include_context = True):

    format_1 = """Use the following context to answer the question.

    Context: {context}

    Question: {question}
    """

    try: 
        context = row["context"]
        question = row["question"]
        answer = row["answer"]

        # FORMAT 1: Teaches the model to use a context to answer (using RAG later)
        if include_context:
            content = format_1.format(context = context, question=question)

            chat = [{"role": "user", "content": content},
            {"role": "assistant", "content": answer}]
        
        # FORMAT 2 : When no context is available: use only the question
        else:
            chat = [{"role": "user", "content": question},
            {"role": "assistant", "content": answer}]

        # format chat
    
        chat = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=False)
    
        return chat

    except Exception as e: 
        print(e)
        return None    




In [9]:
#  test 1
chat = format_row(df2.iloc[0], tokenizer)
print(chat)

# test 2
chat = format_row(df2.iloc[0], tokenizer, include_context=False)
print(chat)


<s> [INST] Use the following context to answer the question.

    Context: Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.

    Question: What area did NVIDIA initially focus on before expanding to other computationally intensive fields?
     [/INST] NVIDIA initially focused on PC graphics.</s>
<s> [INST] What area did NVIDIA initially focus on before expanding to other computationally intensive fields? [/INST] NVIDIA initially focused on PC graphics.</s>


In [10]:
# https://huggingface.co/docs/transformers/main/chat_templating#model-training
# 70 % context-grounded, 30% pure knowledge
import random
from datasets import Dataset

def create_chat_dataset(df, tokenizer):

    # context grounded (70%)
    context_grounded = df
    chat_list = context_grounded.apply(
        lambda x: format_row(x, tokenizer, include_context=True),axis=1
        ).to_list()

    # questions only (30%)
    questions_only = df.sample(n = int(len(df)*0.3/0.7))
    chat_list.extend(
        questions_only.apply(
            lambda x: format_row(x, tokenizer, include_context=False), axis=1
            )
        )

    # dataset
    dataset = Dataset.from_dict({"chat": chat_list})    

    return dataset


In [11]:
dataset = create_chat_dataset(df2, tokenizer)

In [12]:
# split into train, val and test
split_1 = dataset.train_test_split(test_size=0.2, seed = 42)
split_2 = split_1["test"]. train_test_split(test_size=0.5, seed = 42)

dataset_splits = {
    "train": split_1["train"],
    "val": split_2["train"],
    "test": split_2["test"]
}


In [13]:
dataset_splits

{'train': Dataset({
     features: ['chat'],
     num_rows: 8000
 }),
 'val': Dataset({
     features: ['chat'],
     num_rows: 1000
 }),
 'test': Dataset({
     features: ['chat'],
     num_rows: 1000
 })}

## Parameter Efficient Fine Tuning

In [14]:
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
import wandb
wandb.login()

wandb.init(
    project="finance-bro",
    name="qlora-mistral-7b-run1",
    config={
        "model": "mistral-7b-instruct-v0.2",
        "lora_r": 16,
        "lora_alpha": 32,
        "target_modules": "all_7",
        "dataset_mix": "70ctx_30noctx",
        "max_seq_length": 2048,
        "quantization": "nf4_4bit"
    }
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: coffeedrunk to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [16]:
# Quantized 
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# QLoRA config: 4-bit storage, BF16 compute
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4", # NF4 is better than INT4 for LLMs
    bnb_4bit_compute_dtype=torch.bfloat16,  # RTX 5070 Ti supports BF16 natively
    bnb_4bit_use_double_quant= True # quantize the quantization constants too → saves ~0.4GB extra

)

# trained_weights = "mistralai/Mistral-7B-Instruct-v0.2"

# quantized base model
base_model = AutoModelForCausalLM.from_pretrained(
    trained_weights,
    quantization_config=bnb_config,
    device_map = "auto")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [17]:
# https://huggingface.co/docs/transformers/main/peft
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from transformers import AutoTokenizer, MistralForCausalLM


lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM, # type of task to train on
    r = 16, # dimension of the smallest matrix/rank adapter capacity
    lora_alpha = 32, # scaling factor
    lora_dropout= 0.05, # dropout of LoRA layers
    target_modules=["q_proj", "v_proj"],  # which layers get LoRA
    bias = "none", # freeze the bias terms (only train LoRA matrices)
     
)

# prepare model for 4-bit quantization (enables gradient checkpointing and casts the right layers to trainable precision)
base_model = prepare_model_for_kbit_training(base_model) 

# Adapt model for the Lora 
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


trainable params: 6,815,744 || all params: 7,248,547,840 || trainable%: 0.0940


In [23]:
from trl import SFTTrainer , SFTConfig

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"   # required for causal LM training

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_splits["train"],
    eval_dataset=dataset_splits["val"],
    processing_class=tokenizer,
    args=SFTConfig(
        dataset_text_field="chat",
        max_length=2048,
        report_to="wandb",
        output_dir="./finance-bro-checkpoints",
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        num_train_epochs=3,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        load_best_model_at_end=True,
        bf16=True,
    )
)


Adding EOS to train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [24]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss
100,1.407469,1.483248
200,1.413453,1.332529
300,1.126477,1.246225
400,1.141202,1.221710
500,1.232786,1.212054
600,1.140999,1.203100
700,1.174698,1.196162
800,1.213567,1.191754
900,1.316389,1.186876
1000,1.157556,1.182429


TrainOutput(global_step=6000, training_loss=1.160076498190562, metrics={'train_runtime': 6038.8599, 'train_samples_per_second': 3.974, 'train_steps_per_second': 0.994, 'total_flos': 1.7378518693281792e+17, 'train_loss': 1.160076498190562})